In [5]:
# ============================================================
# CELL 1 — INSTALL LIBRARIES
# ============================================================
# This installs the free/open-source libraries we need for
# embeddings, vector search, BM25, and hybrid retrieval.

!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-chroma \
    chromadb \
    rank_bm25 \
    sentence-transformers

In [6]:
# ============================================================
# CELL 2 — IMPORT LIBRARIES
# ============================================================
# This imports the classes needed to create documents,
# embeddings, vector search, and BM25 search.

from langchain_core.documents import Document

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma

from langchain_community.retrievers import BM25Retriever

print("✅ Imports successful")

✅ Imports successful


/tmp/ipykernel_1622/716400014.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [7]:
# ============================================================
# CELL 3 — CREATE SAMPLE DOCUMENTS
# ============================================================
# This creates a few small documents that we can search.
# These are only for learning how retrieval works.

texts = [
    "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",

    "Tesla manufactures electric vehicles including the Model 3, Model Y, and Cybertruck.",

    "Google was founded by Larry Page and Sergey Brin.",

    "SpaceX develops rockets and spacecraft for space exploration.",

    "NVIDIA designs graphics processing units and AI hardware.",

    "Apple develops the iPhone, Mac computers, and other consumer products."
]

documents = [
    Document(page_content=text)
    for text in texts
]

print(f"Created {len(documents)} documents.")

Created 6 documents.


In [8]:
# ============================================================
# CELL 4 — VIEW THE DOCUMENTS
# ============================================================
# This lets us see exactly what information is available
# to the retrieval system.

for i, document in enumerate(documents, start=1):

    print(f"\nDocument {i}")
    print(document.page_content)


Document 1
Microsoft acquired GitHub for 7.5 billion dollars in 2018.

Document 2
Tesla manufactures electric vehicles including the Model 3, Model Y, and Cybertruck.

Document 3
Google was founded by Larry Page and Sergey Brin.

Document 4
SpaceX develops rockets and spacecraft for space exploration.

Document 5
NVIDIA designs graphics processing units and AI hardware.

Document 6
Apple develops the iPhone, Mac computers, and other consumer products.


In [9]:
# ============================================================
# CELL 5 — LOAD FREE EMBEDDING MODEL
# ============================================================
# This model converts text into numerical vectors.
# Similar meanings should produce similar vectors.

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded


In [10]:
# ============================================================
# CELL 6 — SEE AN EMBEDDING
# ============================================================
# This converts one sentence into a numerical vector so you
# can see what an embedding actually looks like.

text = "Tesla makes electric vehicles."

vector = embedding_model.embed_query(text)

print("Number of dimensions:", len(vector))

print("\nFirst 10 values:")
print(vector[:10])

Number of dimensions: 384

First 10 values:
[-0.03689335286617279, 0.06723472476005554, 0.013652544468641281, 0.04488799721002579, -0.00871746800839901, -0.006033171899616718, -0.025270691141486168, 0.02778925932943821, -0.014060763642191887, -0.013749656267464161]


In [11]:
# ============================================================
# CELL 7 — CREATE VECTOR DATABASE
# ============================================================
# This stores the documents and their embeddings so that
# we can perform semantic similarity searches.

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

print("✅ Vector database created")

✅ Vector database created


In [12]:
# ============================================================
# CELL 8 — CREATE VECTOR RETRIEVER
# ============================================================
# This tells the vector database to return the 3 documents
# that are most semantically similar to a query.

vector_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3
    }
)

print("✅ Vector retriever created")

✅ Vector retriever created


In [13]:
# ============================================================
# CELL 9 — TEST VECTOR SEARCH
# ============================================================
# This searches by meaning rather than simply matching
# exact words.

query = "company involved in space exploration"

results = vector_retriever.invoke(query)

print("QUERY:")
print(query)

print("\nRESULTS:")

for i, document in enumerate(results, start=1):

    print(f"\n{i}. {document.page_content}")

QUERY:
company involved in space exploration

RESULTS:

1. SpaceX develops rockets and spacecraft for space exploration.

2. Google was founded by Larry Page and Sergey Brin.

3. Microsoft acquired GitHub for 7.5 billion dollars in 2018.


In [14]:
# ============================================================
# CELL 10 — CREATE BM25 RETRIEVER
# ============================================================
# BM25 is a keyword-based retrieval algorithm.
# It is particularly useful when exact words or names matter.

bm25_retriever = BM25Retriever.from_documents(
    documents
)

bm25_retriever.k = 3

print("✅ BM25 retriever created")

✅ BM25 retriever created


In [15]:
# ============================================================
# CELL 11 — TEST BM25 SEARCH
# ============================================================
# This searches for documents using keyword matching.

query = "Cybertruck"

results = bm25_retriever.invoke(query)

print("QUERY:")
print(query)

print("\nBM25 RESULTS:")

for i, document in enumerate(results, start=1):

    print(f"\n{i}. {document.page_content}")

QUERY:
Cybertruck

BM25 RESULTS:

1. Apple develops the iPhone, Mac computers, and other consumer products.

2. NVIDIA designs graphics processing units and AI hardware.

3. SpaceX develops rockets and spacecraft for space exploration.


In [16]:
# ============================================================
# CELL 12 — COMPARE THE TWO RETRIEVERS
# ============================================================
# This runs the same query through both retrieval methods
# so we can see how their results differ.

query = "electric vehicle Cybertruck"

print("=" * 60)
print("VECTOR SEARCH")
print("=" * 60)

vector_results = vector_retriever.invoke(query)

for i, document in enumerate(vector_results, 1):
    print(f"{i}. {document.page_content}")


print("\n" + "=" * 60)
print("BM25 SEARCH")
print("=" * 60)

bm25_results = bm25_retriever.invoke(query)

for i, document in enumerate(bm25_results, 1):
    print(f"{i}. {document.page_content}")

VECTOR SEARCH
1. Tesla manufactures electric vehicles including the Model 3, Model Y, and Cybertruck.
2. SpaceX develops rockets and spacecraft for space exploration.
3. Apple develops the iPhone, Mac computers, and other consumer products.

BM25 SEARCH
1. Tesla manufactures electric vehicles including the Model 3, Model Y, and Cybertruck.
2. Apple develops the iPhone, Mac computers, and other consumer products.
3. NVIDIA designs graphics processing units and AI hardware.


In [17]:
# ============================================================
# CELL 13 — RECIPROCAL RANK FUSION
# ============================================================
# RRF combines multiple ranked lists.
# A document receives a higher score when it appears near
# the top of one or more retrieval results.

def reciprocal_rank_fusion(
    result_lists,
    k=60
):

    scores = {}

    documents_by_id = {}

    for results in result_lists:

        for rank, document in enumerate(
            results,
            start=1
        ):

            # Use the document text as a simple ID.
            doc_id = document.page_content

            # RRF formula:
            #
            #       1
            #   -----------
            #      k + rank
            #
            score = 1 / (k + rank)

            scores[doc_id] = (
                scores.get(doc_id, 0)
                + score
            )

            documents_by_id[doc_id] = document

    # Sort documents from highest score to lowest.
    ranked_ids = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return [
        documents_by_id[doc_id]
        for doc_id in ranked_ids
    ]

In [18]:
# ============================================================
# CELL 14 — TEST RRF
# ============================================================
# This runs vector search and BM25 independently and then
# combines their ranked results using RRF.

query = "electric vehicle Cybertruck"

vector_results = vector_retriever.invoke(query)

bm25_results = bm25_retriever.invoke(query)

rrf_results = reciprocal_rank_fusion(
    [
        vector_results,
        bm25_results
    ]
)

print("QUERY:")
print(query)

print("\nRRF RESULTS:")

for i, document in enumerate(
    rrf_results,
    start=1
):

    print(f"{i}. {document.page_content}")

QUERY:
electric vehicle Cybertruck

RRF RESULTS:
1. Tesla manufactures electric vehicles including the Model 3, Model Y, and Cybertruck.
2. Apple develops the iPhone, Mac computers, and other consumer products.
3. SpaceX develops rockets and spacecraft for space exploration.
4. NVIDIA designs graphics processing units and AI hardware.


In [28]:
# ============================================================
# CELL 15 — SEE RRF SCORES
# ============================================================
# This version shows how much score each document receives
# during Reciprocal Rank Fusion.

def rrf_with_scores(
    result_lists,
    k=60
):

    scores = {}

    for results in result_lists:

        for rank, document in enumerate(
            results,
            start=1
        ):

            doc_id = document.page_content

            score = 1 / (k + rank)

            scores[doc_id] = (
                scores.get(doc_id, 0)
                + score
            )

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked

In [29]:
# ============================================================
# CELL 16 — PRINT RRF SCORES
# ============================================================
# This displays the final RRF score for each document.

scores = rrf_with_scores(
    [
        vector_results,
        bm25_results
    ]
)

for rank, (document, score) in enumerate(
    scores,
    start=1
):

    print(f"{rank}. Score: {score:.5f}")
    print(f"   {document}\n")

1. Score: 0.03279
   Tesla manufactures electric vehicles including the Model 3, Model Y, and Cybertruck.

2. Score: 0.03200
   Apple develops the iPhone, Mac computers, and other consumer products.

3. Score: 0.01613
   SpaceX develops rockets and spacecraft for space exploration.

4. Score: 0.01587
   NVIDIA designs graphics processing units and AI hardware.



In [30]:
# ============================================================
# CELL 17 — EXPERIMENT WITH QUERIES
# ============================================================
# Try different queries and observe how vector search,
# BM25, and RRF behave differently.

queries = [
    "How much did Microsoft pay for GitHub?",
    "electric vehicle Cybertruck",
    "company involved in space exploration",
    "graphics processing unit AI",
]

for query in queries:

    print("\n" + "=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    vector_results = vector_retriever.invoke(query)

    bm25_results = bm25_retriever.invoke(query)

    rrf_results = reciprocal_rank_fusion(
        [
            vector_results,
            bm25_results
        ]
    )

    print("\nTop RRF result:")

    if rrf_results:
        print(
            rrf_results[0].page_content
        )


QUERY: How much did Microsoft pay for GitHub?

Top RRF result:
Microsoft acquired GitHub for 7.5 billion dollars in 2018.

QUERY: electric vehicle Cybertruck

Top RRF result:
Tesla manufactures electric vehicles including the Model 3, Model Y, and Cybertruck.

QUERY: company involved in space exploration

Top RRF result:
SpaceX develops rockets and spacecraft for space exploration.

QUERY: graphics processing unit AI

Top RRF result:
NVIDIA designs graphics processing units and AI hardware.


In [33]:
# ============================================================
# CELL 18 — CREATE HYBRID RETRIEVER
# ============================================================
# This combines vector search and BM25 search into one
# retriever. We give both methods equal importance.

from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever
    ],
    weights=[
        0.5,   # 50% vector search
        0.5    # 50% BM25 search
    ]
)

print("✅ Hybrid retriever created successfully.")

✅ Hybrid retriever created successfully.


In [34]:
# ============================================================
# CELL 19 — TEST HYBRID RETRIEVAL
# ============================================================
# This sends one query through the combined retriever.

query = "electric vehicle Cybertruck"

results = hybrid_retriever.invoke(query)

print("QUERY:")
print(query)

print("\nHYBRID RESULTS:")

for i, document in enumerate(
    results,
    start=1
):

    print(f"\n{i}. {document.page_content}")

QUERY:
electric vehicle Cybertruck

HYBRID RESULTS:

1. Tesla manufactures electric vehicles including the Model 3, Model Y, and Cybertruck.

2. Apple develops the iPhone, Mac computers, and other consumer products.

3. SpaceX develops rockets and spacecraft for space exploration.

4. NVIDIA designs graphics processing units and AI hardware.


In [35]:
# ============================================================
# CELL 20 — COMPLETE RETRIEVAL PIPELINE
# ============================================================
# This demonstrates the complete process:
# Query → Vector Search + BM25 → RRF → Top Documents.

query = "What company makes GPUs for AI?"

print("QUERY:")
print(query)

# 1. Semantic retrieval.
vector_results = vector_retriever.invoke(query)

# 2. Keyword retrieval.
bm25_results = bm25_retriever.invoke(query)

# 3. Combine the two ranked lists.
final_results = reciprocal_rank_fusion(
    [
        vector_results,
        bm25_results
    ]
)

# 4. Keep only the top 3 documents.
final_results = final_results[:3]

print("\nFINAL RETRIEVED DOCUMENTS:")

for i, document in enumerate(
    final_results,
    start=1
):

    print(
        f"\n{i}. {document.page_content}"
    )

QUERY:
What company makes GPUs for AI?

FINAL RETRIEVED DOCUMENTS:

1. NVIDIA designs graphics processing units and AI hardware.

2. SpaceX develops rockets and spacecraft for space exploration.

3. Google was founded by Larry Page and Sergey Brin.
